# Automatidata project

## Course 4 — Regression Analysis: Simplify Complex Data Relationships

The data consulting firm **Automatidata** is working with the **New York City Taxi and Limousine Commission (NYC TLC)**.

The goal of this project is to build and evaluate a multiple linear regression model that predicts taxi fare amounts using historical trip data collected over the course of a year.


# Course 4 End-of-course project: Build a multiple linear regression model

The project has three main parts:

1. Exploratory data analysis and checking model assumptions.
2. Model building and evaluation.
3. Interpretation of model results.

The analysis follows the PACE framework:

- **Plan**
- **Analyze**
- **Construct**
- **Execute**


## PACE: Plan

### Task 1. Imports and loading

Import the packages needed for data manipulation, visualization, feature engineering, preprocessing, model training, and model evaluation.


In [ ]:
# Imports

# Packages for numerics + dataframes
import pandas as pd
import numpy as np

# Packages for visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Packages for date conversions for calculating trip durations
import datetime as dt

# Packages for MLR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
# Load dataset into dataframe
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Abraham123w/Google-Advanced-Data-Analytics-Professional-Certificate/"
    "main/4.%20Regression%20Analysis%20Simplify%20Complex%20Data%20Relationships/"
    "2017_Yellow_Taxi_Trip_Data.csv"
)

df0 = pd.read_csv(DATA_URL)

print("Dataset cargado correctamente.")
print(f"Filas: {df0.shape[0]:,}")
print(f"Columnas: {df0.shape[1]}")

df0.head()


## PACE: Analyze

Exploratory data analysis is important before building a multiple linear regression model because it helps:

- Validate assumptions.
- Detect outliers.
- Clean data.
- Identify missing values and duplicates.
- Investigate multicollinearity.
- Decide whether features need scaling.


### Task 2a. Explore data with EDA


In [ ]:
# Start with `.shape` and `.info()`
print(df0.shape)
df0.info()


In [ ]:
# Check for missing data
print("Valores faltantes por columna:")
print(df0.isna().sum())

# Check for duplicates
print("\nCantidad de filas duplicadas:")
print(df0.duplicated().sum())


In [ ]:
# Use .describe()
df0.describe()


### Task 2b. Convert pickup and drop-off columns to datetime


In [ ]:
# Check the format of the data
print(df0[['tpep_pickup_datetime', 'tpep_dropoff_datetime']].head())

# Review the current data types
print(df0[['tpep_pickup_datetime', 'tpep_dropoff_datetime']].dtypes)


In [ ]:
# Convert datetime columns to datetime
df0['tpep_pickup_datetime'] = pd.to_datetime(df0['tpep_pickup_datetime'])
df0['tpep_dropoff_datetime'] = pd.to_datetime(df0['tpep_dropoff_datetime'])

# Confirm the conversion
print(df0.dtypes[['tpep_pickup_datetime', 'tpep_dropoff_datetime']])


### Task 2c. Create a duration column

Create a new column called `duration` that represents the total number of minutes each taxi ride took.


In [ ]:
# Create `duration` column
df0['duration'] = (
    df0['tpep_dropoff_datetime'] - df0['tpep_pickup_datetime']
).dt.total_seconds() / 60

# Verify the calculation
df0[['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'duration']].head()


### Outliers

The most relevant columns to inspect for outliers are:

- `trip_distance`
- `fare_amount`
- `duration`


In [ ]:
### YOUR CODE HERE ###
df0.info()


### Task 2d. Box plots


In [ ]:
### YOUR CODE HERE ###
# Create one figure with three box plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.boxplot(data=df0, x='trip_distance', ax=axes[0])
axes[0].set_title('Distancia del Viaje (trip_distance)')

sns.boxplot(data=df0, x='fare_amount', ax=axes[1])
axes[1].set_title('Monto de la Tarifa (fare_amount)')

sns.boxplot(data=df0, x='duration', ax=axes[2])
axes[2].set_title('Duración (duration)')

plt.tight_layout()
plt.show()


All three variables contain outliers.

Negative fares are likely errors, reversals, or disputes. Zero distances or durations may correspond to canceled or invalid trips and provide limited information for predicting legitimate taxi fares.


### Task 2e. Imputations


In [ ]:
# Are trip distances of 0 bad data or very short trips rounded down?
sorted(df0['trip_distance'].unique())[:10]


In [ ]:
### YOUR CODE HERE ###
(df0['trip_distance'] == 0).sum()


#### `fare_amount` outliers


In [ ]:
### YOUR CODE HERE ###

# Descriptive statistics
print(df0['fare_amount'].describe())

# Lowest values
print("\nValores más bajos de la tarifa:")
print(df0['fare_amount'].sort_values().head(10))

# Highest values
print("\nValores más altos de la tarifa:")
print(df0['fare_amount'].sort_values(ascending=False).head(10))


In [ ]:
# Impute values less than $0 with 0
df0.loc[df0['fare_amount'] < 0, 'fare_amount'] = 0

# Verify the new minimum
print(df0['fare_amount'].min())


In [ ]:
def outlier_imputer(column_list, iqr_factor):
    '''
    Impute upper-limit values in specified columns based on their
    interquartile range.

    Arguments:
        column_list: A list of columns to iterate over
        iqr_factor: A number representing x in the formula
                    Q3 + (x * IQR)
    '''
    for col in column_list:
        # Reassign minimum to zero
        df0.loc[df0[col] < 0, col] = 0

        # Calculate upper threshold
        q1 = df0[col].quantile(0.25)
        q3 = df0[col].quantile(0.75)
        iqr = q3 - q1
        upper_threshold = q3 + (iqr_factor * iqr)
        print(col, " threshold: ", upper_threshold)

        # Reassign values above the threshold
        df0.loc[df0[col] > upper_threshold, col] = upper_threshold


# Apply to fare_amount with a factor of 6
outlier_imputer(['fare_amount'], 6)


#### `duration` outliers


In [ ]:
# Call .describe() for duration outliers
df0['duration'].describe()


In [ ]:
# Impute a 0 for any negative values
df0.loc[df0['duration'] < 0, 'duration'] = 0


In [ ]:
# Impute the high outliers
outlier_imputer(['duration'], 6)


### Task 3a. Feature engineering

The model cannot use the actual trip duration at the time a fare must be predicted because the trip has not finished. Instead, route-level historical averages can be used.


#### Create `mean_distance`


In [ ]:
# Create `pickup_dropoff` column
df0['pickup_dropoff'] = (
    df0['PULocationID'].astype(str)
    + ' '
    + df0['DOLocationID'].astype(str)
)


In [ ]:
# Group by pickup_dropoff and calculate the mean trip_distance
grouped = df0.groupby('pickup_dropoff')[['trip_distance']].mean()

# Preview results
grouped.head()


In [ ]:
# 1. Convert `grouped` to a dictionary
grouped_dict = grouped.to_dict()

# 2. Keep only the inner dictionary
grouped_dict = grouped_dict['trip_distance']


In [ ]:
# 1. Create mean_distance as a copy of pickup_dropoff
df0['mean_distance'] = df0['pickup_dropoff']

# 2. Map grouped_dict to mean_distance
df0['mean_distance'] = df0['mean_distance'].map(grouped_dict)

# Confirm that it worked
df0[['pickup_dropoff', 'mean_distance', 'trip_distance']].head()


#### Create `mean_duration`


In [ ]:
# 1. Group by pickup_dropoff and calculate mean duration
grouped_duration = df0.groupby('pickup_dropoff')[['duration']].mean()

# 2. Convert to a simplified dictionary
grouped_duration_dict = grouped_duration.to_dict()['duration']

# 3. Create and map the mean_duration column
df0['mean_duration'] = df0['pickup_dropoff']
df0['mean_duration'] = df0['mean_duration'].map(grouped_duration_dict)

# Confirm that it worked
df0[['pickup_dropoff', 'duration', 'mean_duration']].head()


#### Create day and month columns


In [ ]:
# Ensure datetime type
df0['tpep_pickup_datetime'] = pd.to_datetime(df0['tpep_pickup_datetime'])

# Create day and month columns
df0['day'] = df0['tpep_pickup_datetime'].dt.day_name().str.lower()
df0['month'] = df0['tpep_pickup_datetime'].dt.month_name().str.lower()

df0[['tpep_pickup_datetime', 'day', 'month']].head()


#### Create `rush_hour`

Rush hour is defined as a weekday ride occurring from 06:00–10:00 or 16:00–20:00.


In [ ]:
# Create `rush_hour` column
df0['rush_hour'] = df0['tpep_pickup_datetime'].dt.hour

def make_rush_hour(hour):
    if 6 <= hour < 10 or 16 <= hour < 20:
        return 1
    return 0

df0['rush_hour'] = df0['rush_hour'].apply(make_rush_hour)

# Set weekends to zero
df0.loc[
    df0['day'].isin(['saturday', 'sunday']),
    'rush_hour'
] = 0


In [ ]:
# Apply the `rush_hourizer()` function to the new column
def rush_hourizer(row):
    if 6 <= row['tpep_pickup_datetime'].hour < 10:
        am_rush = 1
    else:
        am_rush = 0

    if 16 <= row['tpep_pickup_datetime'].hour < 20:
        pm_rush = 1
    else:
        pm_rush = 0

    if row['day'] in ['saturday', 'sunday']:
        return 0

    if am_rush == 1 or pm_rush == 1:
        return 1

    return 0

df0['rush_hour'] = df0.apply(rush_hourizer, axis=1)


### Task 4. Scatter plot


In [ ]:
# Visualize the relationship between mean_duration and fare_amount
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x='mean_duration',
    y='fare_amount',
    data=df0,
    alpha=0.5
)

plt.title('Relación entre Duración Promedio y Tarifa')
plt.xlabel('Duración Promedio (minutos)')
plt.ylabel('Monto de la Tarifa ($)')
plt.show()


In [ ]:
# Identify the most frequent fares above $50
df0[df0['fare_amount'] > 50]['fare_amount'].value_counts().head()


In [ ]:
# Display all columns
pd.set_option('display.max_columns', None)

# Examine the first 30 trips with a fare_amount of $52
df0[df0['fare_amount'] == 52].head(30)


The $52 horizontal line represents the JFK flat fare, while $62.50 represents the upper cap imposed during outlier treatment.


### Task 5. Isolate modeling variables


In [ ]:
# Review current columns
df0.info()

# Keep a modeling copy and drop variables that are redundant,
# unavailable at prediction time, or not needed for this model.
df2 = df0.copy()

df2 = df2.drop(
    [
        'Unnamed: 0',
        'tpep_pickup_datetime',
        'tpep_dropoff_datetime',
        'trip_distance',
        'RatecodeID',
        'store_and_fwd_flag',
        'PULocationID',
        'DOLocationID',
        'payment_type',
        'extra',
        'mta_tax',
        'tip_amount',
        'tolls_amount',
        'improvement_surcharge',
        'total_amount',
        'duration',
        'pickup_dropoff',
    ],
    axis=1,
)

df2.info()


### Task 6. Pair plot


In [ ]:
# Pairwise relationships
sns.pairplot(
    df2[['fare_amount', 'mean_duration', 'mean_distance']]
)
plt.show()


### Task 7. Identify correlations


In [ ]:
# Correlation matrix for numeric variables
correlation_matrix = df2.corr(numeric_only=True)
correlation_matrix


In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(
    df2.corr(numeric_only=True),
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)

plt.title('Mapa de Calor de Correlaciones - Dataset df2')
plt.show()


`mean_distance` and `mean_duration` are strongly correlated with `fare_amount`. They are also correlated with one another, so model diagnostics should be interpreted carefully.


## PACE: Construct

### Task 8a. Split data into outcome variable and features


In [ ]:
# Define X and y
X = df2.drop(columns=['fare_amount'])
y = df2['fare_amount']

print("Variables Predictoras (X):")
print(X.head())

print("\nVariable Objetivo (y):")
print(y.head())


### Task 8b. Pre-process data


In [ ]:
# Convert VendorID to string so it is dummy encoded
X['VendorID'] = X['VendorID'].astype(str)

# Dummy encode categorical variables
X = pd.get_dummies(X, drop_first=True, dtype=int)

X.head()


### Split data into training and test sets


In [ ]:
# Create training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0
)


### Standardize the data


In [ ]:
# Instantiate and fit the scaler on training data
scaler = StandardScaler()
scaler.fit(X_train)

# Transform training data
X_train_scaled = scaler.transform(X_train)


### Fit the model


In [ ]:
# Instantiate and fit the linear regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

model


### Task 8c. Evaluate the model on training data


In [ ]:
# Predictions on training data
y_pred_train = model.predict(X_train_scaled)

# Metrics
r2 = r2_score(y_train, y_pred_train)
mae = mean_absolute_error(y_train, y_pred_train)
mse = mean_squared_error(y_train, y_pred_train)
rmse = np.sqrt(mse)
rss = mse * len(y_train)

print('--- Métricas de Entrenamiento ---')
print(f'R^2: {r2:.4f}')
print(f'MAE: {mae:.4f}')
print(f'MSE: {mse:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'RSS: {rss:.4f}')


### Evaluate the model on test data


In [ ]:
# Scale test data using the scaler fit on training data
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Predictions and metrics on test data
y_pred_test = model.predict(X_test_scaled)

r2_test = r2_score(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)

print('--- Métricas de Prueba (Test) ---')
print(f'R^2: {r2_test:.4f}')
print(f'MAE: {mae_test:.4f}')
print(f'RMSE: {rmse_test:.4f}')


## PACE: Execute

### Task 9a. Results


In [ ]:
# Create a results dataframe
results = pd.DataFrame({
    'actual': y_test,
    'predicted': y_pred_test
})

# Calculate residuals
results['residual'] = results['actual'] - results['predicted']

results.head()


### Task 9b. Visualize model results


In [ ]:
# Actual vs predicted
sns.scatterplot(
    x='actual',
    y='predicted',
    data=results,
    alpha=0.5
)

max_val = max(results['actual'].max(), results['predicted'].max())
min_val = min(results['actual'].min(), results['predicted'].min())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    color='red',
    linestyle='--'
)

plt.title('Actual vs. Predicted Fare Amount')
plt.xlabel('Actual Fare ($)')
plt.ylabel('Predicted Fare ($)')
plt.show()


In [ ]:
# Distribution of residuals
sns.histplot(results['residual'], bins=25)
plt.title('Distribution of the residuals')
plt.xlabel('Residual value')
plt.ylabel('Count')
plt.show()


In [ ]:
# Calculate residual mean
residual_mean = results['residual'].mean()
print(f"La media de los residuos es: {residual_mean}")


In [ ]:
# Residuals over predicted values
sns.scatterplot(
    x='predicted',
    y='residual',
    data=results
)

plt.axhline(0, color='red', linestyle='--')
plt.title('Residuos vs. Valores Predichos')
plt.xlabel('Valores Predichos ($)')
plt.ylabel('Residuos ($)')
plt.show()


### Task 9c. Coefficients


In [ ]:
# Feature names and model coefficients
feature_names = X_train.columns

coefficients = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_
})

coefficients['Abs_Coefficient'] = coefficients['Coefficient'].abs()
coefficients = coefficients.sort_values(
    by='Abs_Coefficient',
    ascending=False
)

coefficients[['Feature', 'Coefficient']]


### Task 9d. Conclusion

Key takeaways:

- Exploratory analysis and data preparation are essential before modeling.
- Outliers and invalid values can strongly influence regression coefficients.
- Route-level engineered features improve predictive performance.
- Mean distance is the strongest predictor of fare amount.
- Mean duration is also influential.
- Temporal variables have smaller effects.
- Training and test metrics are similar, suggesting reasonable generalization.
- The model explains a large proportion of fare variation, but results should still be interpreted alongside regression assumptions and operational constraints.


**Congratulations! You have completed the lab.**
